# TrAISformer: Full Pipeline - Data Preprocessing → Training → Inference
## Processing 15-min Interpolated U.S. Maritime AIS Data

This notebook implements the complete TrAISformer pipeline as described in the paper:
1. **Load & Analyze** interpolated parquet data
2. **Preprocess** trajectories (filter, normalize)
3. **Tokenize** features to discrete bins (key innovation)
4. **Create pickle files** in TrAISformer format
5. **Train** the transformer model
6. **Run inference** and evaluate predictions

**Expected Output:** Trajectory predictions up to 10+ hours ahead with <10 nautical mile error

## STEP 1: Import Libraries & Configure Environment

This cell imports all necessary libraries and sets up paths for the TrAISformer pipeline.
- **Pandas/Numpy:** Data manipulation and numerical operations
- **PyTorch:** Deep learning framework for the transformer model
- **Scikit-learn:** Data normalization and utilities
- **Matplotlib/Seaborn:** Visualization
- **Pickle:** Serialization format for TrAISformer data

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pickle
import os
import sys
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Setup paths
WORKSPACE_ROOT = Path(r"F:\PyTorch_GPU\maritime_monitoring_preprocessing")
INTERPOLATED_DATA_PATH = (
    WORKSPACE_ROOT
    / "interpolated_results"
    / "interpolated_ais_data_20200105_20200112_15min.parquet"
)
TRAISFORMER_PATH = WORKSPACE_ROOT / "TRAIS_Former_" / "CEE_TrAISformer"
OUTPUT_DIR = TRAISFORMER_PATH / "data" / "us_maritime"
RESULTS_DIR = TRAISFORMER_PATH / "results"

# Create output directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Add TrAISformer modules to path
sys.path.insert(0, str(TRAISFORMER_PATH))

print(f"✓ Workspace root: {WORKSPACE_ROOT}")
print(f"✓ Data file: {INTERPOLATED_DATA_PATH.exists()}")
print(f"✓ Output directory: {OUTPUT_DIR}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")

✓ Workspace root: F:\PyTorch_GPU\maritime_monitoring_preprocessing
✓ Data file: True
✓ Output directory: F:\PyTorch_GPU\maritime_monitoring_preprocessing\TRAIS_Former_\CEE_TrAISformer\data\us_maritime
✓ PyTorch version: 2.6.0+cu124
✓ CUDA available: True


## STEP 2: Load & Analyze Interpolated Data

This cell loads the parquet file containing 15-minute interpolated AIS data.
- **Parquet format:** Efficient columnar storage
- **Columns analyzed:** LAT, LON, SOG, COG, MMSI, timestamp
- **Output:** Data bounds and statistics needed for normalization in subsequent steps

In [3]:
# Load interpolated data - using enhanced version with SOG and COG calculated
print("Loading interpolated AIS data from parquet (with SOG & COG)...")

# First, try to load the enhanced version; if not available, load original
enhanced_path = (
    INTERPOLATED_DATA_PATH.parent
    / "interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet"
)

if enhanced_path.exists():
    print(f"Loading enhanced parquet with SOG/COG: {enhanced_path.name}")
    df = pd.read_parquet(enhanced_path)
else:
    print(
        f"Enhanced version not found yet. Using original: {INTERPOLATED_DATA_PATH.name}"
    )
    df = pd.read_parquet(INTERPOLATED_DATA_PATH)

df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
df = df.set_index("BaseDateTime").sort_index()

print(f"\n{'='*60}")
print("DATA OVERVIEW")
print(f"{'='*60}")
print(f"Total records: {len(df):,}")
print(f"Unique vessels (MMSI): {df['MMSI'].nunique():,}")
print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())

# Analyze geographic and speed bounds
print(f"\n{'='*60}")
print("DATA STATISTICS")
print(f"{'='*60}")

lat_min, lat_max = df["LAT"].min(), df["LAT"].max()
lon_min, lon_max = df["LON"].min(), df["LON"].max()

# Handle SOG and COG columns
if "SOG" in df.columns:
    sog_min, sog_max = df["SOG"].min(), df["SOG"].max()
else:
    sog_min, sog_max = 0, 30
    print("⚠️  SOG column not found - will calculate in next step")

if "COG" in df.columns:
    cog_min, cog_max = df["COG"].min(), df["COG"].max()
else:
    cog_min, cog_max = 0, 360
    print("⚠️  COG column not found - will calculate in next step")

print(f"\nLatitude:   {lat_min:.4f}° to {lat_max:.4f}° (range: {lat_max-lat_min:.4f}°)")
print(f"Longitude:  {lon_min:.4f}° to {lon_max:.4f}° (range: {lon_max-lon_min:.4f}°)")
print(f"SOG (knots): {sog_min:.2f} to {sog_max:.2f}")
print(f"COG (°):    {cog_min:.2f} to {cog_max:.2f}")

# Store for later use
BOUNDS = {
    "lat_min": lat_min,
    "lat_max": lat_max,
    "lon_min": lon_min,
    "lon_max": lon_max,
    "sog_max": max(sog_max, 30.0),  # Use 30 knots as standard max
    "cog_max": 360.0,
}

print(f"\nNormalization bounds (will use for tokenization):")
for key, val in BOUNDS.items():
    print(f"  {key}: {val}")

# Check data quality
print(f"\n{'='*60}")
print("DATA QUALITY CHECKS")
print(f"{'='*60}")
required_cols = ["LAT", "LON"]
if "SOG" in df.columns:
    required_cols.append("SOG")
if "COG" in df.columns:
    required_cols.append("COG")

print(f"NaN values in required columns:")
print(df[required_cols].isnull().sum())
print(f"\nDuplicate timestamps per vessel: {df.index.duplicated().sum()}")

Loading interpolated AIS data from parquet (with SOG & COG)...
Loading enhanced parquet with SOG/COG: interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet

DATA OVERVIEW
Total records: 6,863,558
Unique vessels (MMSI): 14,496
Date range: 2020-01-05 00:00:01 to 2020-01-13 00:14:50

Columns: ['LAT', 'LON', 'interpolated', 'MMSI', 'SOG', 'COG']

First 5 rows:
                          LAT        LON  interpolated       MMSI  SOG  COG
BaseDateTime                                                               
2020-01-05 00:00:01  13.58120  144.83658         False  369970581  0.0  0.0
2020-01-05 00:00:02  12.44007  144.52146         False  219802000  0.0  0.0
2020-01-05 00:00:04  13.46193  144.66508         False  367796190  0.0  0.0
2020-01-05 00:00:07  13.42761  144.66547         False  368926398  0.0  0.0
2020-01-05 00:00:10  13.45492  144.63816         False  368926395  0.0  0.0

DATA STATISTICS

Latitude:   -2574.2780° to 3275.1834° (range: 5849.4614°)
Longitude:  -10384.6

## Step 2.5: Calculate Derived Features (SOG & COG)

**Objective:** Create enhanced parquet file with interpolated SOG and COG columns
- **Input:** Original parquet with LAT, LON (15-min interpolated)
- **Method:** Calculate Speed Over Ground (SOG) from haversine distance, Course Over Ground (COG) from bearing angles
- **Output:** New parquet file with 6 columns: `[BaseDateTime, LAT, LON, interpolated, MMSI, SOG, COG]`

In [5]:
import numpy as np


def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two points using Haversine formula.
    Returns distance in nautical miles.
    """
    R = 3440.065  # Earth radius in nautical miles

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    distance = R * c

    return distance


def calculate_bearing(lat1, lon1, lat2, lon2):
    """
    Calculate bearing (course) from point 1 to point 2.
    Returns bearing in degrees (0-360).
    """
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1

    y = np.sin(dlon) * np.cos(lat2)
    x = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(y, x))

    # Normalize to 0-360 range
    bearing = (bearing + 360) % 360
    return bearing


# Load original interpolated data (without SOG/COG)
print("Loading original interpolated parquet file...")
df_original = pd.read_parquet(INTERPOLATED_DATA_PATH)
df_original["BaseDateTime"] = pd.to_datetime(df_original["BaseDateTime"])

# Sort by MMSI and timestamp
df_original = df_original.sort_values(["MMSI", "BaseDateTime"]).reset_index(drop=True)

print(
    f"Loaded {len(df_original):,} records from {df_original['MMSI'].nunique():,} vessels"
)

# Calculate SOG and COG for each vessel
print("\nCalculating SOG and COG features...")

sog_values = []
cog_values = []
vessel_ids = df_original["MMSI"].values

for idx in range(len(df_original)):
    if idx == 0:
        # First point: use next point to estimate
        lat1 = df_original.loc[idx, "LAT"]
        lon1 = df_original.loc[idx, "LON"]
        lat2 = df_original.loc[idx + 1, "LAT"]
        lon2 = df_original.loc[idx + 1, "LON"]
        time1 = df_original.loc[idx, "BaseDateTime"]
        time2 = df_original.loc[idx + 1, "BaseDateTime"]
        mmsi1 = vessel_ids[idx]
        mmsi2 = vessel_ids[idx + 1]
    elif idx == len(df_original) - 1:
        # Last point: use previous point
        lat1 = df_original.loc[idx - 1, "LAT"]
        lon1 = df_original.loc[idx - 1, "LON"]
        lat2 = df_original.loc[idx, "LAT"]
        lon2 = df_original.loc[idx, "LON"]
        time1 = df_original.loc[idx - 1, "BaseDateTime"]
        time2 = df_original.loc[idx, "BaseDateTime"]
        mmsi1 = vessel_ids[idx - 1]
        mmsi2 = vessel_ids[idx]
    else:
        # Middle point: use surrounding points for better estimate
        lat1 = df_original.loc[idx - 1, "LAT"]
        lon1 = df_original.loc[idx - 1, "LON"]
        lat2 = df_original.loc[idx + 1, "LAT"]
        lon2 = df_original.loc[idx + 1, "LON"]
        time1 = df_original.loc[idx - 1, "BaseDateTime"]
        time2 = df_original.loc[idx + 1, "BaseDateTime"]
        mmsi1 = vessel_ids[idx - 1]
        mmsi2 = vessel_ids[idx + 1]

    # Only calculate if same vessel (no multi-vessel jumps)
    if mmsi1 == mmsi2 == vessel_ids[idx]:
        time_diff_hours = (time2 - time1).total_seconds() / 3600.0

        if time_diff_hours > 0:
            dist_nm = haversine_distance(lat1, lon1, lat2, lon2)
            sog = dist_nm / time_diff_hours  # knots
            bearing = calculate_bearing(lat1, lon1, lat2, lon2)
        else:
            sog = 0.0
            bearing = 0.0
    else:
        sog = 0.0
        bearing = 0.0

    sog_values.append(sog)
    cog_values.append(bearing)

    if (idx + 1) % 100000 == 0:
        print(f"  Processed {idx + 1:,} records...")

print(f"Completed SOG/COG calculations")

# Add to dataframe and clip to valid ranges
df_original["SOG"] = np.array(sog_values)
df_original["COG"] = np.array(cog_values)

# Clip to valid ranges
df_original["SOG"] = df_original["SOG"].clip(0, 30)  # 0-30 knots
df_original["COG"] = df_original["COG"].clip(0, 360)  # 0-360 degrees

print(f"\nFeature ranges after calculation:")
print(f"  SOG: {df_original['SOG'].min():.2f} - {df_original['SOG'].max():.2f} knots")
print(f"  COG: {df_original['COG'].min():.2f} - {df_original['COG'].max():.2f} degrees")

# Save enhanced parquet file
enhanced_output_path = (
    INTERPOLATED_DATA_PATH.parent
    / "interpolated_ais_data_20200105_20200112_15min_with_sog_cog.parquet"
)
print(f"\nSaving enhanced parquet file: {enhanced_output_path.name}")
df_original.to_parquet(enhanced_output_path)
print(f"✓ Saved {len(df_original):,} records to enhanced parquet file")

# Display sample of enhanced data
print(f"\nSample of enhanced data:")
print(df_original[["BaseDateTime", "LAT", "LON", "SOG", "COG", "MMSI"]].head(10))

Loading original interpolated parquet file...
Loaded 6,863,558 records from 14,496 vessels

Calculating SOG and COG features...
  Processed 100,000 records...
  Processed 200,000 records...
  Processed 300,000 records...
  Processed 400,000 records...
  Processed 500,000 records...
  Processed 600,000 records...
  Processed 700,000 records...
  Processed 800,000 records...
  Processed 900,000 records...
  Processed 1,000,000 records...
  Processed 1,100,000 records...
  Processed 1,200,000 records...
  Processed 1,300,000 records...
  Processed 1,400,000 records...
  Processed 1,500,000 records...
  Processed 1,600,000 records...
  Processed 1,700,000 records...
  Processed 1,800,000 records...
  Processed 1,900,000 records...
  Processed 2,000,000 records...
  Processed 2,100,000 records...
  Processed 2,200,000 records...
  Processed 2,300,000 records...
  Processed 2,400,000 records...
  Processed 2,500,000 records...
  Processed 2,600,000 records...
  Processed 2,700,000 records...

## STEP 3: Trajectory Filtering & Preprocessing

This critical preprocessing step filters raw AIS data into clean trajectories:
1. **Remove stationary vessels** (SOG < 0.05 knots threshold)
2. **Filter by geographic bounds** (valid maritime region)
3. **Remove NaN values** and quality issues
4. **Minimum length check** (at least 36 timesteps = 9 hours of 15-min data)
5. **Create per-vessel trajectories** (chronological order)

**Output:** List of dictionaries with MMSI and normalized trajectory arrays

In [4]:
# Configuration for filtering
MIN_SOG_THRESHOLD = 0.05  # knots - filter stationary vessels
MIN_TRAJECTORY_LENGTH = 36  # timesteps (9 hours at 15-min intervals)
LAT_MIN_BOUNDS = BOUNDS["lat_min"] - 1
LAT_MAX_BOUNDS = BOUNDS["lat_max"] + 1
LON_MIN_BOUNDS = BOUNDS["lon_min"] - 1
LON_MAX_BOUNDS = BOUNDS["lon_max"] + 1

# Preprocess trajectories
trajectories = []
skipped_reasons = {
    "too_short": 0,
    "all_stationary": 0,
    "nan_values": 0,
    "out_of_bounds": 0,
    "valid": 0,
}

print("Processing trajectories by vessel...")
print(f"Filters: min_sog={MIN_SOG_THRESHOLD}, min_length={MIN_TRAJECTORY_LENGTH}")

for mmsi, group in tqdm(df.groupby("MMSI"), desc="Processing vessels"):
    group = group.sort_index()  # Ensure chronological order

    # FILTER 1: Remove stationary vessels
    moving_mask = group["SOG"] > MIN_SOG_THRESHOLD
    if moving_mask.sum() == 0:
        skipped_reasons["all_stationary"] += 1
        continue

    first_moving_idx = moving_mask.idxmax()
    group = group.loc[first_moving_idx:].reset_index(drop=True)

    # FILTER 2: Remove NaN values
    group = group.dropna(subset=["LAT", "LON", "SOG", "COG"]).reset_index(drop=True)
    if len(group) == 0:
        skipped_reasons["nan_values"] += 1
        continue

    # FILTER 3: Check if too short
    if len(group) < MIN_TRAJECTORY_LENGTH:
        skipped_reasons["too_short"] += 1
        continue

    # FILTER 4: Geographic bounds check
    in_bounds = (
        (group["LAT"] >= LAT_MIN_BOUNDS)
        & (group["LAT"] <= LAT_MAX_BOUNDS)
        & (group["LON"] >= LON_MIN_BOUNDS)
        & (group["LON"] <= LON_MAX_BOUNDS)
    )
    group = group[in_bounds].reset_index(drop=True)

    if len(group) < MIN_TRAJECTORY_LENGTH:
        skipped_reasons["out_of_bounds"] += 1
        continue

    # BUILD TRAJECTORY ARRAY: [LAT_norm, LON_norm, SOG_norm, COG_norm, TIMESTAMP_unix]
    traj_array = np.zeros((len(group), 5), dtype=np.float32)

    # Normalize geographic coordinates
    traj_array[:, 0] = (group["LAT"].values - BOUNDS["lat_min"]) / (
        BOUNDS["lat_max"] - BOUNDS["lat_min"]
    )
    traj_array[:, 1] = (group["LON"].values - BOUNDS["lon_min"]) / (
        BOUNDS["lon_max"] - BOUNDS["lon_min"]
    )

    # Normalize speeds and courses
    traj_array[:, 2] = group["SOG"].values / BOUNDS["sog_max"]
    traj_array[:, 3] = group["COG"].values / 360.0

    # Unix timestamp
    traj_array[:, 4] = group.index.astype(np.int64).values // 10**9

    # Clip to [0, 0.9999) to prevent boundary issues during tokenization
    traj_array[:, :4] = np.clip(traj_array[:, :4], 0, 0.9999)

    trajectories.append({"mmsi": int(mmsi), "traj": traj_array})
    skipped_reasons["valid"] += 1

# Print statistics
print(f"\n{'='*60}")
print("TRAJECTORY FILTERING RESULTS")
print(f"{'='*60}")
print(f"Valid trajectories: {skipped_reasons['valid']:,}")
print(f"Skipped (too short): {skipped_reasons['too_short']:,}")
print(f"Skipped (all stationary): {skipped_reasons['all_stationary']:,}")
print(f"Skipped (NaN values): {skipped_reasons['nan_values']:,}")
print(f"Skipped (out of bounds): {skipped_reasons['out_of_bounds']:,}")

print(f"\n{'='*60}")
print("TRAJECTORY STATISTICS")
print(f"{'='*60}")
if trajectories:
    lengths = [len(t["traj"]) for t in trajectories]
    total_timesteps = sum(lengths)
    print(f"Total trajectories: {len(trajectories):,}")
    print(f"Total timesteps: {total_timesteps:,}")
    print(f"Avg trajectory length: {np.mean(lengths):.1f} timesteps")
    print(f"Min trajectory length: {np.min(lengths)} timesteps")
    print(f"Max trajectory length: {np.max(lengths)} timesteps")
else:
    print("⚠️  No valid trajectories found!")

Processing trajectories by vessel...
Filters: min_sog=0.05, min_length=36


Processing vessels: 100%|██████████| 14496/14496 [00:27<00:00, 532.60it/s]


TRAJECTORY FILTERING RESULTS
Valid trajectories: 9,948
Skipped (too short): 1,286
Skipped (all stationary): 3,262
Skipped (NaN values): 0
Skipped (out of bounds): 0

TRAJECTORY STATISTICS
Total trajectories: 9,948
Total timesteps: 4,510,839
Avg trajectory length: 453.4 timesteps
Min trajectory length: 36 timesteps
Max trajectory length: 767 timesteps


## STEP 4: Define Tokenization Configuration

**Key Innovation of TrAISformer:** Discrete representation via tokenization
Instead of predicting continuous values (lat, lon, sog, cog), the model treats each feature as a **classification problem** with discrete bins.

This cell defines:
- **Quantization levels** for each feature (lat, lon, speed, course)
- **Embedding dimensions** for each feature
- **Why discrete?** Better handles multimodal distributions + enables cross-entropy loss

The paper uses uniform quantization: `token_idx = int(normalized_value * num_bins)`

In [5]:
# Tokenization configuration (from TrAISformer paper)
lat_range = BOUNDS["lat_max"] - BOUNDS["lat_min"]
lon_range = BOUNDS["lon_max"] - BOUNDS["lon_min"]

# Quantization levels (number of discrete bins for each feature)
# Reduced for GPU memory constraints
# Use fixed 100x100 grid for geographic area (reduces embedding table size)
lat_size = 100  # Reduced from ~58K to 100 bins (fixed grid)
lon_size = 100  # Reduced from ~169K to 100 bins (fixed grid)
sog_size = 30  # Speed: 0-30 knots (keep as is)
cog_size = 72  # Course: 5° per bin (360/72, keep as is)

# Embedding dimensions (learned representation after tokenization)
# Reduced for GPU memory constraints (4GB VRAM)
n_lat_embd = 64
n_lon_embd = 64
n_sog_embd = 32
n_cog_embd = 32

# Transformer architecture (reduced for memory)
n_head = 4  # Reduced from 8
n_layer = 4  # Reduced from 8
max_seqlen = 120  # Maximum sequence length for training

# Create config dictionary
config = {
    "lat_size": lat_size,
    "lon_size": lon_size,
    "sog_size": sog_size,
    "cog_size": cog_size,
    "n_lat_embd": n_lat_embd,
    "n_lon_embd": n_lon_embd,
    "n_sog_embd": n_sog_embd,
    "n_cog_embd": n_cog_embd,
    "n_embd": n_lat_embd + n_lon_embd + n_sog_embd + n_cog_embd,
    "n_head": n_head,
    "n_layer": n_layer,
    "max_seqlen": max_seqlen,
    "full_vocab_size": lat_size + lon_size + sog_size + cog_size,
}

print("TOKENIZATION CONFIGURATION")
print(f"{'='*60}")
print(f"\nQuantization Levels (bins per feature):")
print(f"  Latitude:  {lat_size} bins (1 bin ≈ {lat_range/lat_size:.4f}°)")
print(f"  Longitude: {lon_size} bins (1 bin ≈ {lon_range/lon_size:.4f}°)")
print(f"  SOG:       {sog_size} bins (1 bin ≈ {30.0/sog_size:.2f} knots)")
print(f"  COG:       {cog_size} bins (1 bin ≈ {360.0/cog_size:.1f}°)")

print(f"\nEmbedding Dimensions:")
print(f"  Latitude:  {n_lat_embd}D")
print(f"  Longitude: {n_lon_embd}D")
print(f"  SOG:       {n_sog_embd}D")
print(f"  COG:       {n_cog_embd}D")
print(f"  Total:     {config['n_embd']}D per timestep")

print(f"\nTransformer Architecture:")
print(f"  Blocks:    {n_layer} layers")
print(f"  Heads:     {n_head} attention heads")
print(f"  Max seq:   {max_seqlen} timesteps")

print(f"\nTotal vocab size: {config['full_vocab_size']} (sum of all bins)")

TOKENIZATION CONFIGURATION

Quantization Levels (bins per feature):
  Latitude:  100 bins (1 bin ≈ 58.4946°)
  Longitude: 100 bins (1 bin ≈ 169.3943°)
  SOG:       30 bins (1 bin ≈ 1.00 knots)
  COG:       72 bins (1 bin ≈ 5.0°)

Embedding Dimensions:
  Latitude:  64D
  Longitude: 64D
  SOG:       32D
  COG:       32D
  Total:     192D per timestep

Transformer Architecture:
  Blocks:    4 layers
  Heads:     4 attention heads
  Max seq:   120 timesteps

Total vocab size: 302 (sum of all bins)


## STEP 5: Create Train/Validation/Test Split

Splits preprocessed trajectories into training, validation, and test sets.
**Important:** Split by vessel (MMSI), NOT by timestep, to prevent temporal leakage!

- **Train:** 80% of vessels (used for model training)
- **Validation:** 10% of vessels (used for hyperparameter tuning)
- **Test:** 10% of vessels (final evaluation with unseen vessels)

In [6]:
# Train/Val/Test split (by vessel, not by timestep)
np.random.seed(42)
n_vessels = len(trajectories)
n_train = int(0.8 * n_vessels)
n_val = int(0.1 * n_vessels)

# Shuffle vessel list
indices = np.arange(n_vessels)
np.random.shuffle(indices)

train_idx = indices[:n_train]
val_idx = indices[n_train : n_train + n_val]
test_idx = indices[n_train + n_val :]

train_data = [trajectories[i] for i in train_idx]
val_data = [trajectories[i] for i in val_idx]
test_data = [trajectories[i] for i in test_idx]

print(f"{'='*60}")
print("TRAIN/VAL/TEST SPLIT (by vessel)")
print(f"{'='*60}")

for phase, data in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    n_traj = len(data)
    total_ts = sum(len(t["traj"]) for t in data)
    avg_len = total_ts / n_traj if n_traj > 0 else 0
    print(f"\n{phase}:")
    print(f"  Vessels: {n_traj:,} ({100*n_traj/n_vessels:.1f}%)")
    print(f"  Total timesteps: {total_ts:,}")
    print(f"  Avg trajectory length: {avg_len:.1f}")

# Save as pickle files
print(f"\n{'='*60}")
print("SAVING PICKLE FILES")
print(f"{'='*60}")

pickle_files = [
    (train_data, OUTPUT_DIR / "us_maritime_train.pkl", "Train"),
    (val_data, OUTPUT_DIR / "us_maritime_valid.pkl", "Validation"),
    (test_data, OUTPUT_DIR / "us_maritime_test.pkl", "Test"),
]

for data, filepath, phase in pickle_files:
    with open(filepath, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"✓ {phase}: {filepath.name}")

print(f"\n✓ All pickle files saved to: {OUTPUT_DIR}")

TRAIN/VAL/TEST SPLIT (by vessel)

Train:
  Vessels: 7,958 (80.0%)
  Total timesteps: 3,606,151
  Avg trajectory length: 453.1

Val:
  Vessels: 994 (10.0%)
  Total timesteps: 455,845
  Avg trajectory length: 458.6

Test:
  Vessels: 996 (10.0%)
  Total timesteps: 448,843
  Avg trajectory length: 450.6

SAVING PICKLE FILES
✓ Train: us_maritime_train.pkl
✓ Validation: us_maritime_valid.pkl
✓ Test: us_maritime_test.pkl

✓ All pickle files saved to: F:\PyTorch_GPU\maritime_monitoring_preprocessing\TRAIS_Former_\CEE_TrAISformer\data\us_maritime


## STEP 6: Define Custom AIS Dataset Class

Custom PyTorch Dataset that:
1. Loads trajectories from pickle files
2. Returns padded sequences of fixed length
3. Provides masks for actual vs padding timesteps
4. Handles tokenization on-the-fly (during training)

This matches the format used in the original TrAISformer implementation.

In [7]:
class AISDataset(Dataset):
    """Custom PyTorch Dataset for AIS trajectories"""

    def __init__(self, l_data, max_seqlen=120, device=torch.device("cpu")):
        """
        Args:
            l_data: list of dicts with 'mmsi' and 'traj' (normalized [0,1))
            max_seqlen: maximum sequence length (pad or truncate to this)
            device: torch device (cpu or cuda)
        """
        self.l_data = l_data
        self.max_seqlen = max_seqlen
        self.device = device

    def __len__(self):
        return len(self.l_data)

    def __getitem__(self, idx):
        """
        Returns:
            seq: (max_seqlen, 4) - normalized trajectory [LAT, LON, SOG, COG]
            mask: (max_seqlen,) - 1 for real data, 0 for padding
            seqlen: actual sequence length before padding
            mmsi: vessel identifier
            time_start: Unix timestamp of first point
        """
        V = self.l_data[idx]
        m_v = V["traj"][:, :4]  # Extract only [LAT, LON, SOG, COG]

        # Clip extreme values
        m_v = np.clip(m_v, 0, 0.9999)

        seqlen = min(len(m_v), self.max_seqlen)
        seq = np.zeros((self.max_seqlen, 4), dtype=np.float32)
        seq[:seqlen, :] = m_v[:seqlen, :]

        seq = torch.tensor(seq, dtype=torch.float32, device=self.device)

        mask = torch.zeros(self.max_seqlen, device=self.device)
        mask[:seqlen] = 1.0

        seqlen = torch.tensor(seqlen, dtype=torch.int32)
        mmsi = torch.tensor(V["mmsi"], dtype=torch.int64)
        time_start = torch.tensor(int(V["traj"][0, 4]), dtype=torch.int64)

        return seq, mask, seqlen, mmsi, time_start


# Test dataset
print("Testing AISDataset class...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
test_dataset = AISDataset(
    train_data[:10], max_seqlen=config["max_seqlen"], device=device
)

seq, mask, seqlen, mmsi, time_start = test_dataset[0]
print(f"\nSample batch:")
print(f"  Sequence shape: {seq.shape} (max_seqlen x 4 features)")
print(f"  Mask shape: {mask.shape}")
print(f"  Actual seqlen: {seqlen}")
print(f"  MMSI: {mmsi}")
print(f"✓ Dataset class working correctly")

Testing AISDataset class...

Sample batch:
  Sequence shape: torch.Size([120, 4]) (max_seqlen x 4 features)
  Mask shape: torch.Size([120])
  Actual seqlen: 120
  MMSI: 366959090
✓ Dataset class working correctly


## STEP 7: Implement Tokenization Functions

This cell implements the **core TrAISformer innovation**: converting continuous coordinates to discrete tokens.

**Process:**
1. Normalize feature to [0, 1) range
2. Multiply by number of bins
3. Take integer part to get token index (0 to num_bins-1)

**Example:** 
- Latitude 40.5° in range [25, 45] → normalized=0.775 → token=0.775*250=194

In [8]:
# Tokenization functions (core TrAISformer innovation)
def tokenize_features(traj_array, config):
    """
    Convert continuous features to discrete tokens.

    Args:
        traj_array: (seq_len, 4) array with normalized features [LAT, LON, SOG, COG]
        config: dict with tokenization configuration

    Returns:
        tokens: (seq_len, 4) array with token indices for each feature
    """
    tokens = np.zeros_like(traj_array, dtype=np.int32)

    # Tokenize each feature independently
    tokens[:, 0] = np.clip(
        (traj_array[:, 0] * config["lat_size"]).astype(np.int32),
        0,
        config["lat_size"] - 1,
    )
    tokens[:, 1] = np.clip(
        (traj_array[:, 1] * config["lon_size"]).astype(np.int32),
        0,
        config["lon_size"] - 1,
    )
    tokens[:, 2] = np.clip(
        (traj_array[:, 2] * config["sog_size"]).astype(np.int32),
        0,
        config["sog_size"] - 1,
    )
    tokens[:, 3] = np.clip(
        (traj_array[:, 3] * config["cog_size"]).astype(np.int32),
        0,
        config["cog_size"] - 1,
    )

    return tokens


def embed_tokens(tokens, config, embedders):
    """
    Convert tokens to embeddings using learned embedding tables.

    Args:
        tokens: (batch_size, seq_len, 4) token indices
        config: tokenization config
        embedders: dict of nn.Embedding layers for each feature

    Returns:
        embeddings: (batch_size, seq_len, total_embd_dim)
    """
    batch_size, seq_len = tokens.shape[:2]

    # Convert to torch tensors if needed
    if isinstance(tokens, np.ndarray):
        tokens = torch.from_numpy(tokens).long()

    # Embed each feature
    lat_emb = embedders["lat"](tokens[..., 0])  # (batch, seq_len, n_lat_embd)
    lon_emb = embedders["lon"](tokens[..., 1])  # (batch, seq_len, n_lon_embd)
    sog_emb = embedders["sog"](tokens[..., 2])  # (batch, seq_len, n_sog_embd)
    cog_emb = embedders["cog"](tokens[..., 3])  # (batch, seq_len, n_cog_embd)

    # Concatenate all embeddings
    embeddings = torch.cat([lat_emb, lon_emb, sog_emb, cog_emb], dim=-1)
    return embeddings


# Test tokenization on sample trajectory
print("Testing tokenization functions...")
sample_traj = train_data[0]["traj"][:, :4]  # Get first trajectory
tokens = tokenize_features(sample_traj, config)

print(f"\nSample trajectory (first 5 timesteps):")
print(f"  Normalized values:\n{sample_traj[:5]}")
print(f"\n  Tokenized indices:\n{tokens[:5]}")
print(f"\nToken ranges:")
print(
    f"  LAT tokens: {tokens[:, 0].min()} to {tokens[:, 0].max()} (max: {config['lat_size']-1})"
)
print(
    f"  LON tokens: {tokens[:, 1].min()} to {tokens[:, 1].max()} (max: {config['lon_size']-1})"
)
print(
    f"  SOG tokens: {tokens[:, 2].min()} to {tokens[:, 2].max()} (max: {config['sog_size']-1})"
)
print(
    f"  COG tokens: {tokens[:, 3].min()} to {tokens[:, 3].max()} (max: {config['cog_size']-1})"
)
print(f"\n✓ Tokenization working correctly")

Testing tokenization functions...

Sample trajectory (first 5 timesteps):
  Normalized values:
[[0.44517735 0.6074309  0.00646968 0.03305841]
 [0.4451777  0.6074309  0.03955438 0.03380736]
 [0.445179   0.607431   0.03298815 0.03332404]
 [0.44517908 0.607431   0.00287864 0.028765  ]
 [0.4451791  0.607431   0.00179331 0.01517624]]

  Tokenized indices:
[[44 60  0  2]
 [44 60  1  2]
 [44 60  0  2]
 [44 60  0  2]
 [44 60  0  1]]

Token ranges:
  LAT tokens: 44 to 44 (max: 99)
  LON tokens: 60 to 60 (max: 99)
  SOG tokens: 0 to 29 (max: 29)
  COG tokens: 0 to 71 (max: 71)

✓ Tokenization working correctly


## STEP 8: Build TrAISformer Model (Paper Architecture)

Implements the transformer architecture from the TrAISformer paper:
- **Token Embeddings:** Learned embeddings for each discrete token
- **Transformer Encoder:** Multi-head self-attention with causal masking
- **Classification Heads:** One head per feature for next-token prediction
- **Cross-Entropy Loss:** Multimodal trajectory prediction



In [9]:
import torch.nn as nn


class TrAISformerModel(nn.Module):
    """
    TrAISformer: Transformer-based trajectory prediction model from the paper.

    Paper: "TrAISformer: Multi-modal Transformer Relying on AIS Data for Vessel
    Trajectory Prediction"

    Architecture:
    - Token embeddings for each feature (LAT, LON, SOG, COG)
    - Positional encoding
    - Transformer encoder with causal self-attention
    - Classification heads for next-token prediction
    """

    def __init__(self, config):
        super().__init__()
        self.config = config

        # Embedding layers (one per feature)
        self.lat_embedding = nn.Embedding(config["lat_size"], config["n_lat_embd"])
        self.lon_embedding = nn.Embedding(config["lon_size"], config["n_lon_embd"])
        self.sog_embedding = nn.Embedding(config["sog_size"], config["n_sog_embd"])
        self.cog_embedding = nn.Embedding(config["cog_size"], config["n_cog_embd"])

        # Positional encoding (learns position embeddings)
        self.pos_embedding = nn.Embedding(config["max_seqlen"], config["n_embd"])

        # Dropout
        self.dropout = nn.Dropout(0.1)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config["n_embd"],
            nhead=config["n_head"],
            dim_feedforward=config["n_embd"] * 4,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config["n_layer"],
            norm=nn.LayerNorm(config["n_embd"]),
        )

        # Output heads (one per feature for multi-modal prediction)
        self.lat_head = nn.Linear(config["n_embd"], config["lat_size"])
        self.lon_head = nn.Linear(config["n_embd"], config["lon_size"])
        self.sog_head = nn.Linear(config["n_embd"], config["sog_size"])
        self.cog_head = nn.Linear(config["n_embd"], config["cog_size"])

    def forward(self, tokens, mask=None):
        """
        Forward pass.

        Args:
            tokens: (batch_size, seq_len, 4) with token indices [LAT, LON, SOG, COG]
            mask: (batch_size, seq_len) optional mask (1=valid, 0=padding)

        Returns:
            dict with 'lat', 'lon', 'sog', 'cog' logits each (batch, seq_len, bins)
        """
        batch_size, seq_len, _ = tokens.shape
        device = tokens.device

        # Embed each feature and concatenate
        lat_emb = self.lat_embedding(tokens[..., 0])  # (B, T, n_lat_embd)
        lon_emb = self.lon_embedding(tokens[..., 1])
        sog_emb = self.sog_embedding(tokens[..., 2])
        cog_emb = self.cog_embedding(tokens[..., 3])

        # Concatenate embeddings
        x = torch.cat([lat_emb, lon_emb, sog_emb, cog_emb], dim=-1)  # (B, T, n_embd)

        # Add positional encoding
        positions = torch.arange(seq_len, device=device, dtype=torch.long)
        x = x + self.pos_embedding(positions).unsqueeze(0)

        # Dropout
        x = self.dropout(x)

        # Create causal attention mask (lower triangular)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=device) * float("-inf"), diagonal=1
        )

        # Create padding mask if provided
        src_key_padding_mask = None
        if mask is not None:
            src_key_padding_mask = mask == 0  # (batch, seq_len)

        # Apply transformer encoder
        x = self.transformer_encoder(
            x, mask=causal_mask, src_key_padding_mask=src_key_padding_mask
        )

        # Classification heads
        logits = {
            "lat": self.lat_head(x),  # (B, T, lat_size)
            "lon": self.lon_head(x),
            "sog": self.sog_head(x),
            "cog": self.cog_head(x),
        }

        return logits


# Create and initialize model
print("Creating TrAISformer model (paper architecture)...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrAISformerModel(config).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n{'='*60}")
print("TrAISformer MODEL CREATED")
print(f"{'='*60}")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Device: {device}")
print(f"\nArchitecture:")
print(
    f"  Embeddings: LAT({config['lat_size']}→{config['n_lat_embd']}D), "
    f"LON({config['lon_size']}→{config['n_lon_embd']}D), "
    f"SOG({config['sog_size']}→{config['n_sog_embd']}D), "
    f"COG({config['cog_size']}→{config['n_cog_embd']}D)"
)
print(f"  Total embedding: {config['n_embd']}D")
print(f"  Transformer: {config['n_layer']} layers, {config['n_head']} heads")
print(f"  Max sequence length: {config['max_seqlen']} timesteps")
print(f"\n✓ Model ready for training")

Creating TrAISformer model (paper architecture)...

TrAISformer MODEL CREATED
Total parameters: 1,877,230
Trainable parameters: 1,877,230
Device: cuda

Architecture:
  Embeddings: LAT(100→64D), LON(100→64D), SOG(30→32D), COG(72→32D)
  Total embedding: 192D
  Transformer: 4 layers, 4 heads
  Max sequence length: 120 timesteps

✓ Model ready for training


## STEP 9: Training Loop

Train TrAISformer on maritime trajectory data:
- **Loss:** Cross-entropy for each feature (multimodal next-token prediction)
- **Optimizer:** Adam with cosine annealing learning rate schedule
- **Early Stopping:** Monitor validation loss, save best model
- **GPU:** Uses CUDA for efficient training



In [ ]:
# Prepare data loaders
print("Creating data loaders...")
batch_size = 16
num_workers = 2

train_dataset = AISDataset(train_data, max_seqlen=config["max_seqlen"], device=device)
val_dataset = AISDataset(val_data, max_seqlen=config["max_seqlen"], device=device)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
)

print(f"✓ Train batches: {len(train_loader)} ({len(train_data)} samples)")
print(f"✓ Val batches: {len(val_loader)} ({len(val_data)} samples)")


def train_epoch(model, train_loader, optimizer, device, config):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc="Training", leave=True)
    for batch_idx, (seq, mask, seqlen, mmsi, time_start) in enumerate(pbar):
        # Move to device
        seq = seq.to(device)
        mask = mask.to(device)

        # Tokenize
        batch_size_actual = seq.shape[0]
        tokens = torch.zeros(
            batch_size_actual, seq.shape[1], 4, dtype=torch.int64, device=device
        )

        tokens[:, :, 0] = torch.clamp(
            (seq[:, :, 0] * config["lat_size"]).long(), 0, config["lat_size"] - 1
        )
        tokens[:, :, 1] = torch.clamp(
            (seq[:, :, 1] * config["lon_size"]).long(), 0, config["lon_size"] - 1
        )
        tokens[:, :, 2] = torch.clamp(
            (seq[:, :, 2] * config["sog_size"]).long(), 0, config["sog_size"] - 1
        )
        tokens[:, :, 3] = torch.clamp(
            (seq[:, :, 3] * config["cog_size"]).long(), 0, config["cog_size"] - 1
        )

        # Forward pass
        logits = model(tokens, mask=mask)

        # Compute loss (cross-entropy for each feature, equally weighted)
        loss = torch.tensor(0.0, device=device)
        weight = 0.25  # 1/4 for 4 features

        feature_indices = {"lat": 0, "lon": 1, "sog": 2, "cog": 3}
        feature_sizes = {
            "lat": config["lat_size"],
            "lon": config["lon_size"],
            "sog": config["sog_size"],
            "cog": config["cog_size"],
        }

        for feature, idx in feature_indices.items():
            # Predict next token: logits[t] predicts tokens[t+1]
            pred = logits[feature][:, :-1, :]  # (batch, seq_len-1, vocab_size)
            target = tokens[:, 1:, idx]  # (batch, seq_len-1)

            # Mask out padding
            mask_valid = mask[:, 1:]  # (batch, seq_len-1)

            # Compute cross-entropy
            ce_loss = nn.functional.cross_entropy(
                pred.reshape(-1, feature_sizes[feature]),
                target.reshape(-1),
                reduction="none",
            )

            # Apply mask
            ce_loss = (ce_loss.reshape_as(target) * mask_valid).sum() / (
                mask_valid.sum() + 1e-8
            )
            loss = loss + weight * ce_loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / max(num_batches, 1)


def validate(model, val_loader, device, config):
    """Validate model"""
    model.eval()
    total_loss = 0.0
    num_batches = 0

    with torch.no_grad():
        pbar = tqdm(val_loader, desc="Validating", leave=True)
        for seq, mask, seqlen, mmsi, time_start in pbar:
            seq = seq.to(device)
            mask = mask.to(device)

            # Tokenize
            batch_size_actual = seq.shape[0]
            tokens = torch.zeros(
                batch_size_actual, seq.shape[1], 4, dtype=torch.int64, device=device
            )

            tokens[:, :, 0] = torch.clamp(
                (seq[:, :, 0] * config["lat_size"]).long(), 0, config["lat_size"] - 1
            )
            tokens[:, :, 1] = torch.clamp(
                (seq[:, :, 1] * config["lon_size"]).long(), 0, config["lon_size"] - 1
            )
            tokens[:, :, 2] = torch.clamp(
                (seq[:, :, 2] * config["sog_size"]).long(), 0, config["sog_size"] - 1
            )
            tokens[:, :, 3] = torch.clamp(
                (seq[:, :, 3] * config["cog_size"]).long(), 0, config["cog_size"] - 1
            )

            # Forward pass
            logits = model(tokens, mask=mask)

            # Compute loss
            loss = torch.tensor(0.0, device=device)
            weight = 0.25

            feature_indices = {"lat": 0, "lon": 1, "sog": 2, "cog": 3}
            feature_sizes = {
                "lat": config["lat_size"],
                "lon": config["lon_size"],
                "sog": config["sog_size"],
                "cog": config["cog_size"],
            }

            for feature, idx in feature_indices.items():
                pred = logits[feature][:, :-1, :]
                target = tokens[:, 1:, idx]
                mask_valid = mask[:, 1:]

                ce_loss = nn.functional.cross_entropy(
                    pred.reshape(-1, feature_sizes[feature]),
                    target.reshape(-1),
                    reduction="none",
                )

                ce_loss = (ce_loss.reshape_as(target) * mask_valid).sum() / (
                    mask_valid.sum() + 1e-8
                )
                loss = loss + weight * ce_loss

            total_loss += loss.item()
            num_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / max(num_batches, 1)


# Training hyperparameters
num_epochs = 5
learning_rate = 1e-3
warmup_epochs = 1

# Optimizer and scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Training
print(f"\n{'='*60}")
print("TRAINING TrAISformer")
print(f"{'='*60}")
print(f"Epochs: {num_epochs} | Batch size: {batch_size} | LR: {learning_rate}")
print(f"Device: {device}\n")

history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
patience = 3
patience_counter = 0

for epoch in range(num_epochs):
    print(f"\n{'─'*60}")
    print(f"EPOCH {epoch + 1}/{num_epochs}")
    print(f"{'─'*60}")

    # Train
    train_loss = train_epoch(model, train_loader, optimizer, device, config)

    # Validate
    val_loss = validate(model, val_loader, device, config)

    # Update history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    # Learning rate scheduling
    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"\nTrain Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.2e}"
    )

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save best model
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "config": config,
            },
            RESULTS_DIR / "best_model.pt",
        )
        print(f"✓ BEST MODEL SAVED (val_loss: {val_loss:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n✗ Early stopping (no improvement for {patience} epochs)")
            break

print(f"\n{'='*60}")
print("TRAINING COMPLETED")
print(f"{'='*60}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total epochs trained: {len(history['train_loss'])}")

# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history["train_loss"], label="Train Loss", marker="o", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", marker="s", linewidth=2)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("TrAISformer Training History", fontsize=14, fontweight="bold")
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"✓ Training plot saved to: {RESULTS_DIR / 'training_history.png'}")

SyntaxError: invalid decimal literal (3584963842.py, line 79)

## STEP 10: Inference & Evaluation

Generate trajectory predictions and evaluate on test set:
- **Method:** Autoregressive generation (greedy/argmax sampling)
- **Metric:** Haversine distance error in nautical miles
- **Horizons:** 1h, 3h, 6h, 10h (as in paper)
- **Target:** <10 NM error at 10-hour horizon



In [ ]:
# Load best model for inference
print("Loading best model for inference...")
checkpoint = torch.load(RESULTS_DIR / "best_model.pt", map_location=device)
model = TrAISformerModel(checkpoint["config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(
    f"✓ Model loaded (epoch {checkpoint['epoch']}, val_loss={checkpoint['val_loss']:.4f})"
)


def predict_next_tokens(model, tokens, device, config):
    """
    Predict next tokens given current tokens.

    Args:
        model: TrAISformer model
        tokens: (seq_len, 4) tensor of token indices
        device: torch device
        config: config dict

    Returns:
        next_tokens: (4,) array of next token indices [lat, lon, sog, cog]
    """
    with torch.no_grad():
        tokens_batch = tokens.unsqueeze(0).to(device)  # (1, seq_len, 4)
        logits = model(tokens_batch)  # dict of (1, seq_len, vocab_size)

        next_tokens = []
        for feature in ["lat", "lon", "sog", "cog"]:
            # Get logits for last timestep
            last_logits = logits[feature][0, -1, :]  # (vocab_size,)
            next_token = torch.argmax(last_logits).item()
            next_tokens.append(next_token)

        return np.array(next_tokens)


def denormalize_state(state_norm, bounds):
    """Denormalize state from [0,1) back to original range"""
    state = np.copy(state_norm)
    state[0] = (
        state_norm[0] * (bounds["lat_max"] - bounds["lat_min"]) + bounds["lat_min"]
    )
    state[1] = (
        state_norm[1] * (bounds["lon_max"] - bounds["lon_min"]) + bounds["lon_min"]
    )
    state[2] = state_norm[2] * bounds["sog_max"]
    state[3] = state_norm[3] * 360.0
    return state


def haversine_dist(lat1, lon1, lat2, lon2):
    """Haversine distance in nautical miles"""
    R = 3440.065
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


# Evaluate on test set
print(f"\n{'='*60}")
print("EVALUATING ON TEST SET")
print(f"{'='*60}")

horizons = {
    "1h": 4,
    "3h": 12,
    "6h": 24,
    "10h": 40,
}

results = {h: {"errors": []} for h in horizons}
num_samples = min(50, len(test_data))  # Evaluate on 50 test trajectories

print(f"\nGenerating predictions for {num_samples} test trajectories...\n")

for traj_idx in tqdm(range(num_samples), desc="Inference"):
    traj_norm = test_data[traj_idx]["traj"][:, :4]  # (seq_len, 4)

    # Use first 30 timesteps as context
    context_len = min(30, len(traj_norm) - max(horizons.values()))
    if context_len < 10:
        continue

    # For each prediction horizon
    for horizon_name, horizon_steps in horizons.items():
        if context_len + horizon_steps >= len(traj_norm):
            continue

        # Ground truth
        gt_idx = context_len + horizon_steps - 1
        gt_state_norm = traj_norm[gt_idx]
        gt_state = denormalize_state(gt_state_norm, BOUNDS)

        # Predict
        context = torch.from_numpy(traj_norm[:context_len]).float()

        # Tokenize context
        tokens = torch.zeros(context_len, 4, dtype=torch.int64, device=device)
        tokens[:, 0] = torch.clamp(
            (context[:, 0] * config["lat_size"]).long(), 0, config["lat_size"] - 1
        )
        tokens[:, 1] = torch.clamp(
            (context[:, 1] * config["lon_size"]).long(), 0, config["lon_size"] - 1
        )
        tokens[:, 2] = torch.clamp(
            (context[:, 2] * config["sog_size"]).long(), 0, config["sog_size"] - 1
        )
        tokens[:, 3] = torch.clamp(
            (context[:, 3] * config["cog_size"]).long(), 0, config["cog_size"] - 1
        )

        # Autoregressive prediction
        for step in range(horizon_steps):
            next_tokens = predict_next_tokens(model, tokens, device, config)

            # Denormalize tokens
            next_state_norm = np.array(
                [
                    next_tokens[0] / config["lat_size"],
                    next_tokens[1] / config["lon_size"],
                    next_tokens[2] / config["sog_size"],
                    next_tokens[3] / config["cog_size"],
                ]
            )

            # Append to tokens
            tokens = torch.cat(
                [tokens, torch.from_numpy(next_tokens).unsqueeze(0).long().to(device)],
                dim=0,
            )

            # Keep only last max_seqlen tokens to avoid OOM
            if tokens.shape[0] > config["max_seqlen"]:
                tokens = tokens[-config["max_seqlen"] :].to(device)

        # Final prediction (last denormalized state)
        pred_state = denormalize_state(next_state_norm, BOUNDS)

        # Calculate error
        error_nm = haversine_dist(
            gt_state[0], gt_state[1], pred_state[0], pred_state[1]
        )
        results[horizon_name]["errors"].append(error_nm)

# Print evaluation results
print(f"\n{'='*60}")
print("EVALUATION RESULTS")
print(f"{'='*60}\n")

results_summary = []
for horizon_name in sorted(horizons.keys(), key=lambda x: horizons[x]):
    errors = np.array(results[horizon_name]["errors"])
    if len(errors) > 0:
        mean_err = np.mean(errors)
        median_err = np.median(errors)
        std_err = np.std(errors)
        max_err = np.max(errors)

        print(f"{horizon_name:>4} Prediction Horizon:")
        print(f"      Mean Error:   {mean_err:7.2f} NM")
        print(f"      Median Error: {median_err:7.2f} NM")
        print(f"      Std Dev:      {std_err:7.2f} NM")
        print(f"      Max Error:    {max_err:7.2f} NM")
        print(f"      Samples:      {len(errors)}")
        print()

        results_summary.append(
            {
                "horizon": horizon_name,
                "mean_error": mean_err,
                "median_error": median_err,
                "std_error": std_err,
                "max_error": max_err,
                "samples": len(errors),
            }
        )

# Save results
results_df = pd.DataFrame(results_summary)
results_df.to_csv(RESULTS_DIR / "evaluation_results.csv", index=False)
print(f"✓ Results saved to: {RESULTS_DIR / 'evaluation_results.csv'}")

# Plot error distribution
if results_summary:
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    axes = axes.flatten()

    for idx, (horizon_name, horizon_steps) in enumerate(
        sorted(horizons.items(), key=lambda x: horizons[x])
    ):
        errors = np.array(results[horizon_name]["errors"])
        if len(errors) > 0:
            axes[idx].hist(
                errors, bins=20, color="steelblue", edgecolor="black", alpha=0.7
            )
            axes[idx].axvline(
                np.mean(errors),
                color="red",
                linestyle="--",
                linewidth=2.5,
                label=f"Mean: {np.mean(errors):.2f} NM",
            )
            axes[idx].axvline(
                10, color="green", linestyle=":", linewidth=2, label="Target: 10 NM"
            )
            axes[idx].set_xlabel("Error (Nautical Miles)", fontsize=11)
            axes[idx].set_ylabel("Frequency", fontsize=11)
            axes[idx].set_title(
                f"{horizon_name} Prediction Error", fontsize=12, fontweight="bold"
            )
            axes[idx].legend(fontsize=10)
            axes[idx].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "error_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()

    print(f"✓ Error distribution plot saved: {RESULTS_DIR / 'error_distribution.png'}")

print(f"\n{'='*60}")
print("EVALUATION COMPLETE")
print(f"{'='*60}")

## STEP 8: Load/Create TrAISformer Model

This step loads or creates the transformer model for trajectory prediction.
The model architecture:
- **Token Embeddings:** One embedding table per feature (LAT, LON, SOG, COG)
- **Transformer Encoder:** Multi-head self-attention with causal masking
- **Decoder Heads:** One classification head per feature for next-token prediction
- **Loss Function:** Cross-entropy for each feature, summed


## STEP 9: Training Loop

Train the TrAISformer model on the training set with validation monitoring.
- **Loss Function:** Cross-entropy for each feature (LAT, LON, SOG, COG), equally weighted
- **Optimizer:** Adam with learning rate scheduling
- **Validation:** Evaluate every N epochs to monitor overfitting


## STEP 10: Inference & Evaluation

Generate trajectory predictions and evaluate performance using:
- **Haversine Distance:** Error in nautical miles (main metric)
- **Horizons:** 1h, 3h, 6h, 10h ahead (as in paper)
- **Test Set:** Evaluate on held-out vessels
